Ce TP s'inscrit dans l'exercice 2 du TD1 et utilise le modèle de la question 2.

Les cinq premières composantes de l'état indiquent si les emplacements sont libres (1) ou occupés (0), et la sixième est le chiffre à placer. Les actions sont numérotées de 1 (unités) à 5 (dizaines de milliers). Les paiements ne sont pas escomptés ($\gamma=1$).

In [1]:
import random

def get_initial_state():
    return [1,1,1,1,1, random.randint(0,9)]

**Question**: Écrire une fonction qui renvoie `r,s_`, qui correspondent au paiement et à l'état suivant tirés selon la distribution $p(\,\cdot\,|s,a)$.

In [2]:
def sigma(s,a):
    number = s[:5]
    number[a-1] = 0
    return number

def transition(s,a):
    if s[a-1] == 1:
        return (10**(a-1))*s[5], [*sigma(s,a),random.randint(0,9)]
    else:
        return 0, s

**Question**: Implémenter la politique qui renvoie l'action tirée uniformément parmi les emplacements disponibles.

In [3]:
def pi(s):
    a = random.randint(1,5)
    if s[:5] == [0,0,0,0,0]:
        return a
    else:
        while s[a-1] == 0:
            a = random.randint(1,5)
        return a

**Question**: Compléter la fonction `run_episode`.

In [4]:
def run_episode(policy):
    s = get_initial_state()
    total_reward = 0
    while s[:5] != [0,0,0,0,0]:
        r,s_ = transition(s,policy(s))
        total_reward += r
        s = s_

    return total_reward

**Question**: Évaluer la politique $\pi$ en faisant tourner un grand nombre d'épisodes. Préciser la graine utilisée et le nombre d'épisodes, puis estimer l'erreur standard de la moyenne. Comparer cette moyenne à l'espérance calculée dans le TD1.

L'espérance calculée dans le TD1 vaut $49\,999{,}5$. Une moyenne empirique ne lui est pas nécessairement égale. L'erreur standard est estimée par $s_{G}/\sqrt{N}$, où $s_{G}$ est l'écart-type empirique corrigé des gains de $N$ épisodes.

In [5]:
import statistics

def evaluate(policy, nb=10000, seed=0):
    random.seed(seed)
    rewards = [run_episode(policy) for _ in range(nb)]
    mean = statistics.mean(rewards)
    se = statistics.stdev(rewards) / nb**0.5
    print(f"graine={seed}, épisodes={nb}, moyenne={mean:.4f}, erreur standard={se:.4f}")
    return mean, se

mean_pi, se_pi = evaluate(pi)

graine=0, épisodes=10000, moyenne=49821.6389, erreur standard=288.8148


**Question**: Proposer une meilleure politique $\pi'$ et l'évaluer. Reprendre le même protocole d'évaluation.

On propose la politique qui place les chiffres supérieurs à 7 (resp. inférieurs à 2) dans l'emplacement correspondant à la plus grande puissance de 10 disponible (resp. la plus petite), et les autres chiffres sont placés sur un emplacement disponible tiré uniformément.

In [6]:
def pi_(s):
    if s[:5] == [0,0,0,0,0]:
        return 1
    elif s[5] >= 7:
        a = 5
        while s[a-1]==0:
            a -= 1
        return a
    elif s[5] <= 2:
        a = 1
        while s[a-1]==0:
            a += 1
        return a
    else:
        return pi(s)
    
mean_pi_, se_pi_ = evaluate(pi_)

graine=0, épisodes=10000, moyenne=74304.8281, erreur standard=213.8452


**Vérifications de la correction.**

In [7]:
from unittest.mock import patch

# Placement dans un emplacement libre, sans modifier l'état fourni.
s = [1, 0, 1, 0, 0, 7]
with patch("random.randint", return_value=3):
    assert transition(s, 3) == (700, [1, 0, 0, 0, 0, 3])
assert s == [1, 0, 1, 0, 0, 7]
assert transition(s, 2) == (0, s)

# Dernier placement et états terminaux, conformément au noyau du TD1.
with patch("random.randint", return_value=7) as draw:
    assert transition([0, 0, 0, 0, 1, 9], 5) == (90000, [0, 0, 0, 0, 0, 7])
draw.assert_called_once_with(0, 9)
terminal = [0, 0, 0, 0, 0, 7]
for a in range(1, 6):
    assert transition(terminal, a) == (0, terminal)

# Les chiffres 1 à 5 placés dans les emplacements 1 à 5 donnent 54321.
actions = iter(range(1, 6))
with patch("random.randint", side_effect=[1, 2, 3, 4, 5, 0]):
    assert run_episode(lambda s: next(actions)) == 54321

assert pi_([1, 0, 1, 0, 1, 2]) == 1
assert pi_([1, 0, 1, 0, 1, 7]) == 5
print("Vérifications réussies.")

Vérifications réussies.
